# RNN基础实现




## 实现代码


In [ ]:
import math
import torch
from torch import nn
from torch.nn import functional as F
import re
import collections

# 1. 使用简单的测试数据替代下载逻辑
def load_data_test(batch_size, num_steps):
    # 直接使用一段文本作为测试数据
    text = "hello world. " * 100
    lines = [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in text.split('\n')]
    tokens = [char for line in lines for char in line]

    counter = collections.Counter(tokens)
    token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    vocab_list = ['<unk>'] + [token for token, freq in token_freqs]
    token_to_idx = {token: idx for idx, token in enumerate(vocab_list)}
    idx_to_token = {idx: token for idx, token in enumerate(vocab_list)}
    corpus = [token_to_idx.get(token, 0) for token in tokens]

    class Vocab:
        def __init__(self, idx_to_token, token_to_idx):
            self.idx_to_token = idx_to_token
            self.token_to_idx = token_to_idx
        def __getitem__(self, tokens):
            if not isinstance(tokens, (list, tuple)):
                return self.token_to_idx.get(tokens, 0)
            return [self.token_to_idx.get(token, 0) for token in tokens]
        def __len__(self):
            return len(self.idx_to_token)

    return Vocab(idx_to_token, token_to_idx), torch.tensor(corpus, dtype=torch.long)

# 2. Data Iterator
def get_train_iter(corpus_t, batch_size, num_steps):
    offset = torch.randint(0, num_steps, (1,)).item()
    corpus_t = corpus_t[offset:]
    num_windows = (len(corpus_t) - 1) // num_steps
    windows = corpus_t[:num_windows * num_steps + 1].unfold(0, num_steps + 1, num_steps)
    all_X = windows[:, :-1]
    all_Y = windows[:, 1:]
    perm = torch.randperm(len(all_X))
    all_X, all_Y = all_X[perm], all_Y[perm]
    num_batches = len(all_X) // batch_size
    for i in range(num_batches):
        yield all_X[i*batch_size:(i+1)*batch_size], all_Y[i*batch_size:(i+1)*batch_size]

# 3. Model Definition
class RNNModel(nn.Module):
    def __init__(self, vocab_size, num_hiddens):
        super(RNNModel,self).__init__()
        self.rnn = nn.RNN(vocab_size, num_hiddens)
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.linear = nn.Linear(num_hiddens, vocab_size)

    def forward(self, inputs, state):
        X = F.one_hot(inputs.T, self.vocab_size).type(torch.float32)
        Y, state = self.rnn(X, state)
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state

    def begin_state(self, device, batch_size=1):
        return torch.zeros((self.rnn.num_layers, batch_size, self.num_hiddens), device=device)

# 4. Helper Functions
def predict_ch8(prefix, num_preds, net, vocab, device):
    state = net.begin_state(device, batch_size=1)
    outputs = [vocab[prefix[0]]]
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape((1, 1))
    for y in prefix[1:]:
        _, state = net(get_input(), state)
        outputs.append(vocab[y])
    for _ in range(num_preds):
        y, state = net(get_input(), state)
        outputs.append(int(y.argmax(dim=1).reshape(1)))
    return ''.join([vocab.idx_to_token[i] for i in outputs])

def grad_clipping(net, theta):
    params = [p for p in net.parameters() if p.requires_grad]
    norm = torch.sqrt(sum(torch.sum((p.grad ** 2)) for p in params))
    if norm > theta:
        for param in params: param.grad[:] *= theta / norm

# 5. Training
def train_ch8(net, vocab, corpus_t, lr, num_epochs, device, batch_size, num_steps):
    loss = nn.CrossEntropyLoss()
    updater = torch.optim.SGD(net.parameters(), lr)
    for epoch in range(num_epochs):
        train_iter = get_train_iter(corpus_t, batch_size, num_steps)
        state, total_loss, total_tokens = None, 0, 0
        for X, Y in train_iter:
            if state is None:
                state = net.begin_state(device, batch_size=X.shape[0])
            else:
                state.detach_()
            y = Y.T.reshape(-1)
            X, y = X.to(device), y.to(device)
            y_hat, state = net(X, state)
            l = loss(y_hat, y.long()).mean()
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
            total_loss += l.item() * y.numel()
            total_tokens += y.numel()
        if (epoch + 1) % 50 == 0:
            ppl = math.exp(total_loss / total_tokens)
            print(f'epoch {epoch+1}, perplexity {ppl:.2f}')
            print(predict_ch8('hello', 20, net, vocab, device))

# Execution
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size, num_steps = 16, 10
vocab, corpus_tensor = load_data_test(batch_size, num_steps)
net = RNNModel(len(vocab), 128).to(device)
train_ch8(net, vocab, corpus_tensor, lr=1, num_epochs=200, device=device, batch_size=batch_size, num_steps=num_steps)

epoch 50, perplexity 1.01
hello world hello world h
epoch 100, perplexity 1.01
hello world hello world h
epoch 150, perplexity 1.02
hello world hello world h
epoch 200, perplexity 1.09
hello world hello world h


## 比较两种采样

### 验证与重构过程 (Verification & Rethinking)

1.  **矛盾点分析**：当前代码的核心矛盾在于 `data_Iterator` 使用了 `torch.randperm` 打乱了批次顺序（**随机采样**），而 `train` 函数却使用了 `state.detach_()` 继承了上一个物理批次的状态（**连续状态**）。
2.  **学术严谨性要求**：在学术界或严谨的算法复现中，逻辑链条必须是闭环的。状态的流转必须与数据的物理形态完全对应。
3.  **重构方向**：要消除这个悖论，必须做“二选一”的纯粹化拆解。
    * **路线 A**：坚持“随机采样”，那么状态必须彻底失忆。
    * **路线 B**：坚持“状态连续”，那么数据必须严格序列化，绝对不能打乱。

以下是具体的代码修改方案。你可以根据你的实验目的，选择其中一条路线进行修改。

### 具体代码实现

### 路线 A：走向“纯粹的随机采样” (Pure Random Sampling)

如果你希望模型在训练时看到的数据组合尽可能多样化（保留 `data_Iterator` 中的 `randperm` 打乱机制），那么你必须在训练函数中**切断状态的继承**。

**修改目标**：`train` 函数（对应你截图中的 `train_ch8` 或 `train`）。

**修改方法**：删除对 `state` 是否存在的判断，在每次读取新批次时，**强制生成一个全零的新状态**。

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import math
import re
import collections

# 1. 使用简单的测试数据替代下载逻辑
def load_data_test(batch_size, num_steps):
    # 直接使用一段文本作为测试数据
    text = "hello world. " * 100
    lines = [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in text.split('\n')]
    tokens = [char for line in lines for char in line]

    counter = collections.Counter(tokens)
    token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    vocab_list = ['<unk>'] + [token for token, freq in token_freqs]
    token_to_idx = {token: idx for idx, token in enumerate(vocab_list)}
    idx_to_token = {idx: token for idx, token in enumerate(vocab_list)}
    corpus = [token_to_idx.get(token, 0) for token in tokens]

    class Vocab:
        def __init__(self, idx_to_token, token_to_idx):
            self.idx_to_token = idx_to_token
            self.token_to_idx = token_to_idx
        def __getitem__(self, tokens):
            if not isinstance(tokens, (list, tuple)):
                return self.token_to_idx.get(tokens, 0)
            return [self.token_to_idx.get(token, 0) for token in tokens]
        def __len__(self):
            return len(self.idx_to_token)

    return Vocab(idx_to_token, token_to_idx), torch.tensor(corpus, dtype=torch.long)

# 2. Data Iterator
def data_Iterator(corpus_t, batch_size, num_steps):
  offset=torch.randint(0,num_steps,(1,)).item()
  corpus_t=corpus_t[offset:]
  num_windows=(len(corpus_t)-1)//num_steps
  windows=corpus_t[:num_steps*num_windows+1].unfold(0,num_steps+1,num_steps)
  all_X=windows[:,:-1]
  all_Y=windows[:,1:]
  perm=torch.randperm(len(all_X))
  all_X,all_Y=all_X[perm],all_Y[perm]
  num_batches=len(all_X)//batch_size
  for i in range(num_batches):
    yield all_X[i*batch_size:(i+1)*batch_size],all_Y[i*batch_size:(i+1)*batch_size]

# 3. Model Definition
class RNN(nn.Module):
  def __init__(self,vocab_size,num_hiddens,device):
    super(RNN,self).__init__()
    self.rnn=nn.RNN(input_size=vocab_size,hidden_size=num_hiddens)
    self.vocab_size=vocab_size
    self.num_hiddens=num_hiddens
    self.linear=nn.Linear(in_features=num_hiddens,out_features=vocab_size)
  def forward(self,inputs,state):
    X=F.one_hot(inputs.T,self.vocab_size).type(torch.float32)
    Y,state=self.rnn(X,state)
    outputs=self.linear(Y.reshape(-1,Y.shape[-1]))
    return outputs,state
  def begin_state(self, device, batch_size):
    return torch.zeros((self.rnn.num_layers, batch_size, self.num_hiddens), device=device)

# 4. Helper Functions
def predict(prefix,num_pred,net,vocab,device):
  state=net.begin_state(batch_size=1,device=device)
  outputs=[vocab[prefix[0]]] # Use vocab directly for the first character
  get_input=lambda: torch.tensor([outputs[-1]],device=device).reshape((1,1))
  for y in prefix[1:]:
    _,state=net(get_input(),state)
    outputs.append(vocab[y])
  for _ in range(num_pred):
    y,state=net(get_input(),state)
    outputs.append(y.argmax(dim=1).item())
  return ''.join([vocab.idx_to_token[i] for i in outputs])

def grad_clipping(net,theta):
  params=[p for p in net.parameters() if p.requires_grad]
  norm=torch.sqrt(sum([torch.sum(p.grad**2) for p in params]))
  if norm>theta:
    for param in params:
      param.grad[:]*=theta/norm

# 学术严谨修改版：train 函数内部的数据迭代循环
# 注意: 这里的 get_train_iter 指的是你原本的 get_train_iter 或 data_Iterator，保留其随机打乱的特性。

def train_random_sampling(net, get_train_iter_func, vocab, corpus_t, num_steps, batch_size, lr, num_epochs, device):
    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr)
    for epoch in range(num_epochs):
        train_iter = get_train_iter_func(corpus_t, batch_size, num_steps)
        total_loss, total_tokens = 0, 0

        for X, Y in train_iter:
            # 【修改点】无论是不是第一个批次，全部强制重置状态（彻底切断噪声记忆）
            state = net.begin_state(device, batch_size=X.shape[0])

            y = Y.T.reshape(-1)
            X, y = X.to(device), y.to(device)
            y_hat, state = net(X, state)
            l = loss(y_hat, y.long()).mean()
            optimizer.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            optimizer.step()
            total_loss += l.item() * y.numel()
            total_tokens += y.numel()
        if (epoch + 1) % 50 == 0:
            ppl = math.exp(total_loss / total_tokens)
            print(f'epoch {epoch+1}, perplexity {ppl:.2f}')
            print(predict('hello', 25, net, vocab, device))

# 示例执行 (如果选择此路线，替换你的原始 train 调用)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size, num_steps = 32, 3
vocab, corpus_t = load_data_test(batch_size, num_steps)
hidden_size = 128
net_A = RNN(len(vocab), hidden_size, device).to(device) # 使用新的网络实例避免冲突
train_random_sampling(net_A, data_Iterator, vocab, corpus_t, num_steps, batch_size, 0.1, 500, device=device)

epoch 50, perplexity 1.25
hello world hello world hello 
epoch 100, perplexity 1.15
hello world hello wd world hel
epoch 150, perplexity 1.23
hello wd world hello wd world 
epoch 200, perplexity 1.13
hello world hello world hello 
epoch 250, perplexity 1.24
hello world hello world hello 
epoch 300, perplexity 1.23
hello world hello wd world hel
epoch 350, perplexity 1.12
hello world hello world hello 
epoch 400, perplexity 1.13
hello world hello world hello 
epoch 450, perplexity 1.26
hello world hello world hello 
epoch 500, perplexity 1.13
hello world hello world hello 


### 路线 B：走向“纯粹的顺序划分” (Sequential Partitioning)

这是目前训练大规模语言模型（如 GPT 早期、Transformer 等）更主流的做法。我们需要彻底重写数据迭代器，让数据在各个 Batch 之间呈现**完美的时序连续性**，从而发挥 `state.detach_()` （即 Truncated BPTT）的真正威力。

**修改目标**：重写 `data_Iterator`。

**修改原理**：不再使用 `unfold` 和打乱。而是把整篇长文直接切分成 `$batch_size` 那么多个并行的时间线（行）。每次向右滑动 `$num_steps` 的长度读取数据。

In [ ]:
# 学术严谨修改版：全新的连续采样迭代器
def seq_data_Iterator(corpus_t, batch_size, num_steps):
    # 1. 依然保留初始的随机偏移，增加起始点的多样性
    offset = torch.randint(0, num_steps, (1,)).item()
    corpus_t = corpus_t[offset:]

    # 2. 截断无法被 batch_size 整除的尾部数据，确保矩阵可以完美 Reshape
    num_tokens = ((len(corpus_t) - 1) // batch_size) * batch_size
    Xs = corpus_t[:num_tokens]
    Ys = corpus_t[1:num_tokens + 1] # Y 整体向后错开一位

    # 3. 【核心点】将一维长文本直接折叠成 batch_size 行的矩阵
    # 这样每一行就是一个独立且绝对连续的时间线
    Xs = Xs.reshape(batch_size, -1)
    Ys = Ys.reshape(batch_size, -1)

    # 4. 沿时间线方向，每次切割 num_steps 的窗口，绝对不打乱顺序
    num_batches = Xs.shape[1] // num_steps
    for i in range(num_batches):
        X_batch = Xs[:, i * num_steps : (i + 1) * num_steps]
        Y_batch = Ys[:, i * num_steps : (i + 1) * num_steps]
        yield X_batch, Y_batch

**搭配的训练函数修改**：
如果你使用了上面这个 `seq_data_Iterator`，那么你的 `train` 函数必须**严格保留**原有的逻辑，即只有 Epoch 的第一个批次清空状态，后续批次使用 `detach_()` 继承前一刻的真实记忆：

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import math
import re
import collections

# 1. 使用简单的测试数据替代下载逻辑
def load_data_test(batch_size, num_steps):
    # 直接使用一段文本作为测试数据
    text = "hello world. " * 100
    lines = [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in text.split('\n')]
    tokens = [char for line in lines for char in line]

    counter = collections.Counter(tokens)
    token_freqs = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    vocab_list = ['<unk>'] + [token for token, freq in token_freqs]
    token_to_idx = {token: idx for idx, token in enumerate(vocab_list)}
    idx_to_token = {idx: token for idx, token in enumerate(vocab_list)}
    corpus = [token_to_idx.get(token, 0) for token in tokens]

    class Vocab:
        def __init__(self, idx_to_token, token_to_idx):
            self.idx_to_token = idx_to_token
            self.token_to_idx = token_to_idx
        def __getitem__(self, tokens):
            if not isinstance(tokens, (list, tuple)):
                return self.token_to_idx.get(tokens, 0)
            return [self.token_to_idx.get(token, 0) for token in tokens]
        def __len__(self):
            return len(self.idx_to_token)

    return Vocab(idx_to_token, token_to_idx), torch.tensor(corpus, dtype=torch.long)

# 2. Data Iterator (from original notebook, but not directly used in Route B's training)
def data_Iterator(corpus_t, batch_size, num_steps):
  offset=torch.randint(0,num_steps,(1,)).item()
  corpus_t=corpus_t[offset:]
  num_windows=(len(corpus_t)-1)//num_steps
  windows=corpus_t[:num_steps*num_windows+1].unfold(0,num_steps+1,num_steps)
  all_X=windows[:,:-1]
  all_Y=windows[:,1:]
  perm=torch.randperm(len(all_X))
  all_X,all_Y=all_X[perm],all_Y[perm]
  num_batches=len(all_X)//batch_size
  for i in range(num_batches):
    yield all_X[i*batch_size:(i+1)*batch_size],all_Y[i*batch_size:(i+1)*batch_size]

# Academic rigorous modified version: a new continuous sampling iterator
def seq_data_Iterator(corpus_t, batch_size, num_steps):
    # 1. Retain the initial random offset to increase the diversity of starting points
    offset = torch.randint(0, num_steps, (1,)).item()
    corpus_t = corpus_t[offset:]

    # 2. Truncate tail data that cannot be divided by batch_size to ensure the matrix can be perfectly reshaped
    num_tokens = ((len(corpus_t) - 1) // batch_size) * batch_size
    Xs = corpus_t[:num_tokens]
    Ys = corpus_t[1:num_tokens + 1] # Y is shifted by one position

    # 3. [Core point] Fold the one-dimensional long text directly into a matrix with batch_size rows
    # This way, each row is an independent and absolutely continuous timeline
    Xs = Xs.reshape(batch_size, -1)
    Ys = Ys.reshape(batch_size, -1)

    # 4. Along the timeline direction, cut a window of num_steps each time, absolutely no scrambling
    num_batches = Xs.shape[1] // num_steps
    for i in range(num_batches):
        X_batch = Xs[:, i * num_steps : (i + 1) * num_steps]
        Y_batch = Ys[:, i * num_steps : (i + 1) * num_steps]
        yield X_batch, Y_batch

# 3. Model Definition
class RNN(nn.Module):
  def __init__(self,vocab_size,num_hiddens,device):
    super(RNN,self).__init__()
    self.rnn=nn.RNN(input_size=vocab_size,hidden_size=num_hiddens)
    self.vocab_size=vocab_size
    self.num_hiddens=num_hiddens
    self.linear=nn.Linear(in_features=num_hiddens,out_features=vocab_size)
  def forward(self,inputs,state):
    X=F.one_hot(inputs.T,self.vocab_size).type(torch.float32)
    Y,state=self.rnn(X,state)
    outputs=self.linear(Y.reshape(-1,Y.shape[-1]))
    return outputs,state
  def begin_state(self, device, batch_size):
    return torch.zeros((self.rnn.num_layers, batch_size, self.num_hiddens), device=device)

# 4. Helper Functions
def predict(prefix,num_pred,net,vocab,device):
  state=net.begin_state(batch_size=1,device=device)
  outputs=[vocab[prefix[0]]] # Use vocab directly for the first character
  get_input=lambda: torch.tensor([outputs[-1]],device=device).reshape((1,1))
  for y in prefix[1:]:
    _,state=net(get_input(),state)
    outputs.append(vocab[y])
  for _ in range(num_pred):
    y,state=net(get_input(),state)
    outputs.append(y.argmax(dim=1).item())
  return ''.join([vocab.idx_to_token[i] for i in outputs])

def grad_clipping(net,theta):
  params=[p for p in net.parameters() if p.requires_grad]
  norm=torch.sqrt(sum([torch.sum(p.grad**2) for p in params]))
  if norm>theta:
    for param in params:
      param.grad[:]*=theta/norm

# 当使用顺序划分迭代器时，保留原有的严谨状态流转
def train_sequential_sampling(net, get_train_iter_func, vocab, corpus_t, num_steps, batch_size, lr, num_epochs, device):
    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr)
    for epoch in range(num_epochs):
        train_iter = get_train_iter_func(corpus_t, batch_size, num_steps)
        state = None # 在每个 epoch 开始时重置状态
        total_loss, total_tokens = 0, 0

        for X, Y in train_iter:
            if state is None:
                # 只在 Epoch 刚开始时清空记忆
                state = net.begin_state(device, batch_size=X.shape[0])
            else:
                # 后续批次完美继承真实的上下文，只截断计算图
                state.detach_()

            y = Y.T.reshape(-1)
            X, y = X.to(device), y.to(device)
            y_hat, state = net(X, state)
            l = loss(y_hat, y.long()).mean()
            optimizer.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            optimizer.step()
            total_loss += l.item() * y.numel()
            total_tokens += y.numel()
        if (epoch + 1) % 50 == 0:
            ppl = math.exp(total_loss / total_tokens)
            print(f'epoch {epoch+1}, perplexity {ppl:.2f}')
            print(predict('hello', 25, net, vocab, device))

# 示例执行 (如果选择此路线，替换你的原始 train 调用)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size, num_steps = 32, 3
vocab, corpus_t = load_data_test(batch_size, num_steps)
hidden_size = 128
net_B = RNN(len(vocab), hidden_size, device).to(device) # 使用新的网络实例避免冲突
train_sequential_sampling(net_B, seq_data_Iterator, vocab, corpus_t, num_steps, batch_size, 0.1, 500, device=device)

epoch 50, perplexity 1.04
hello world hello world hello 
epoch 100, perplexity 1.03
hello world hello world hello 
epoch 150, perplexity 1.02
hello world hello world hello 
epoch 200, perplexity 1.02
hello world hello world hello 
epoch 250, perplexity 1.02
hello world hello world hello 
epoch 300, perplexity 1.02
hello world hello world hello 
epoch 350, perplexity 1.02
hello world hello world hello 
epoch 400, perplexity 1.02
hello world hello world hello 
epoch 450, perplexity 1.02
hello world hello world hello 
epoch 500, perplexity 1.02
hello world hello world hello 


### 总结建议

在实际的 AI 学术研究中，**强烈建议采用路线 B（顺序划分 + 状态截断）**。因为保留长距离的真实上下文记忆，是发挥循环神经网络（RNN / LSTM / GRU）时序特性的最核心基础。

## 解读代码细节

### 解读re.sub

In [ ]:
import re

# 待处理字符串：包含数字和标点
text = "User123 said: 'Hello, World!'"

# 场景 A: 只要字母，其余都换成空格
# [^A-Za-z] 表示匹配非字母字符，+ 表示匹配连续多个
clean_text = re.sub('[^A-Za-z]+', ' ', text)
print(f"原文本: {text}")
print(f"清理后: {clean_text.strip()}")

# 场景 B: 隐藏手机号中间四位
phone = "13812345678"
masked_phone = re.sub('(\d{3})\d{4}(\d{4})', r'\1****\2', phone)
print(f"\n脱敏手机号: {masked_phone}")

原文本: User123 said: 'Hello, World!'
清理后: User said Hello World

脱敏手机号: 138****5678


<>:14: SyntaxWarning: invalid escape sequence '\d'
<>:14: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_3141/3484223039.py:14: SyntaxWarning: invalid escape sequence '\d'
  masked_phone = re.sub('(\d{3})\d{4}(\d{4})', r'\1****\2', phone)


### re和collection包的展示

In [ ]:
import re
import collections

# re 示例：提取单词
text = "Hello, world! 123."
clean_text = re.sub('[^A-Za-z ]+', '', text)
print(f"原始文本: {text}")
print(f"清洗后: {clean_text}")

# collections 示例：统计频率
chars = ['a', 'b', 'a', 'c', 'b', 'a']
counter = collections.Counter(chars)
print(f"字符统计: {dict(counter)}")
print(f"出现最多的字符: {counter.most_common(1)}")

原始文本: Hello, world! 123.
清洗后: Hello world 
字符统计: {'a': 3, 'b': 2, 'c': 1}
出现最多的字符: [('a', 3)]


In [ ]:
import collections

# 模拟一些字符统计结果
my_counter = collections.Counter("apple")
print(f"原始 items: {list(my_counter.items())}")

# 按频率（次数）从大到小排序
sorted_items = sorted(my_counter.items(), key=lambda x: x[1], reverse=True)
print(f"排序后: {sorted_items}")

原始 items: [('a', 1), ('p', 2), ('l', 1), ('e', 1)]
排序后: [('p', 2), ('a', 1), ('l', 1), ('e', 1)]


### enumerate的用法

In [ ]:
fruits = ['apple', 'banana', 'cherry']

print("使用 enumerate 遍历：")
for i, fruit in enumerate(fruits):
    print(f"索引 {i} 对应的是: {fruit}")

# 也可以直接转化为列表查看结构
print(f"\nenumerate 的内部结构: {list(enumerate(fruits))}")

使用 enumerate 遍历：
索引 0 对应的是: apple
索引 1 对应的是: banana
索引 2 对应的是: cherry

enumerate 的内部结构: [(0, 'apple'), (1, 'banana'), (2, 'cherry')]


### 解读字典中的get()

In [ ]:
fruits={"apple","banana","orange"}
fruits_dic=enumerate(fruits)
print(f"字典: {dict(fruits_dic)}")
fruit_to_idx={fruit:idx for idx,fruit in fruits_dic}
print(f"映射: {dict(fruit_to_idx)}")

字典: {0: 'banana', 1: 'orange', 2: 'apple'}
映射: {}


#### 💡 为什么 `fruit_to_idx` 是空的？

`enumerate` 返回的是一个**迭代器**。当你执行 `dict(fruits_dic)` 时，迭代器已经被消耗完了。我们可以通过以下方式修正：

In [ ]:
fruits = {"apple", "banana", "orange"}

# 方案 1：将其转化为列表，这样可以多次使用
fruits_list = list(enumerate(fruits))
print(f"列表形式: {fruits_list}")

# 第一次使用
print(f"字典 1: {dict(fruits_list)}")

# 第二次使用：现在不会为空了
fruit_to_idx = {fruit: idx for idx, fruit in fruits_list}
print(f"映射结果: {fruit_to_idx}")

fruit_idx=[fruit_to_idx.get(char,0) for char in fruit]
print(f"水果的idx：{fruit_to_idx}")
print(type(fruit_idx))

列表形式: [(0, 'banana'), (1, 'orange'), (2, 'apple')]
字典 1: {0: 'banana', 1: 'orange', 2: 'apple'}
映射结果: {'banana': 0, 'orange': 1, 'apple': 2}
水果的idx：{'banana': 0, 'orange': 1, 'apple': 2}
<class 'list'>


### 🔍 数据迭代器 (`get_train_iter`) 原理演示
运行以下代码，直观查看 `unfold` 和 `offset` 是如何工作的。

In [ ]:
import torch

# 假设我们的语料是 0 到 19 的数字，num_steps(序列长度)设为 5
test_corpus = torch.arange(20)
steps = 5

print(f"原始语料: {test_corpus}")

# 1. 模拟 unfold 操作 (不加 offset)
# 窗口大小为 steps + 1 = 6， 步幅为 5
windows = test_corpus.unfold(0, steps + 1, steps)
print(f"\nunfold 生成的窗口矩阵:\n{windows}")

# 2. 模拟切分 X 和 Y
X = windows[:, :-1]
Y = windows[:, 1:]
print(f"\n特征 X (前5位):\n{X}")
print(f"标签 Y (后5位，即 X 的预测目标):\n{Y}")

# 3. 模拟打乱
perm = torch.randperm(len(X))
print(f"\n随机打乱后的索引: {perm}")
print(f"打乱后的第一个 Batch 的 X:\n{X[perm][0]}")

原始语料: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19])

unfold 生成的窗口矩阵:
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 5,  6,  7,  8,  9, 10],
        [10, 11, 12, 13, 14, 15]])

特征 X (前5位):
tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14]])
标签 Y (后5位，即 X 的预测目标):
tensor([[ 1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10],
        [11, 12, 13, 14, 15]])

随机打乱后的索引: tensor([1, 2, 0])
打乱后的第一个 Batch 的 X:
tensor([5, 6, 7, 8, 9])


### 🎲 `offset` 随机偏移原理演示
运行以下代码，观察同一段语料在不同 `offset` 下是如何被切分的。

In [ ]:
import torch

def demo_offset(offset_val):
    test_data = torch.arange(12)  # 原始语料 0-11
    steps = 3

    # 模拟代码中的偏移逻辑
    shifted_data = test_data[offset_val:]

    # 使用 unfold 切分窗口
    windows = shifted_data.unfold(0, steps + 1, steps)

    print(f"当 offset = {offset_val} 时:")
    print(f"偏移后的语料: {shifted_data.tolist()}")
    print(f"生成的窗口 (X+Y):\n{windows}\n")

# 演示两种不同的随机偏移结果
demo_offset(0) # 不偏移
demo_offset(2) # 随机跳过前两个字符

当 offset = 0 时:
偏移后的语料: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
生成的窗口 (X+Y):
tensor([[0, 1, 2, 3],
        [3, 4, 5, 6],
        [6, 7, 8, 9]])

当 offset = 2 时:
偏移后的语料: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
生成的窗口 (X+Y):
tensor([[ 2,  3,  4,  5],
        [ 5,  6,  7,  8],
        [ 8,  9, 10, 11]])



### 📏 `unfold` 丢弃剩余元素演示
观察当总长度不是窗口长度整数倍时会发生什么。

In [ ]:
import torch

# 准备 7 个元素: [0, 1, 2, 3, 4, 5, 6]
data = torch.arange(7)

# 设置窗口大小为 3，步幅(stride)也为 3
# 预期：
# 窗口 1: [0, 1, 2]
# 窗口 2: [3, 4, 5]
# 剩余: [6] -> 长度只有 1，不足 3

windows = data.unfold(dimension=0, size=3, step=3)

print(f"原始数据: {data.tolist()}")
print(f"窗口大小: 3, 步幅: 3")
print(f"生成的窗口:\n{windows}")
print(f"\n注意：元素 '6' 因为凑不齐一个窗口被丢弃了。")

原始数据: [0, 1, 2, 3, 4, 5, 6]
窗口大小: 3, 步幅: 3
生成的窗口:
tensor([[0, 1, 2],
        [3, 4, 5]])

注意：元素 '6' 因为凑不齐一个窗口被丢弃了。


### 🔢 `data_Iterator` 分批次过程的具体例子

让我们假设 `corpus_t` 经过偏移、`unfold` 和打乱后，我们得到了以下 `all_X` 和 `all_Y`，并且设置 `batch_size = 3`，`num_steps = 4`。

**模拟数据:**
*   `all_X` 包含 7 个序列，每个序列有 4 个元素。
*   `all_Y` 对应 `all_X` 的目标序列，也包含 7 个序列，每个序列 4 个元素。

In [ ]:
import torch

# 模拟经过处理（偏移、unfold、打乱）后的 all_X 和 all_Y
# 这里假设总共有 7 个序列，每个序列长度为 4
all_X_example = torch.tensor([
    [1, 2, 3, 4],    # 序列 0
    [5, 6, 7, 8],    # 序列 1
    [9, 10, 11, 12], # 序列 2
    [13, 14, 15, 16],# 序列 3
    [17, 18, 19, 20],# 序列 4
    [21, 22, 23, 24],# 序列 5
    [25, 26, 27, 28] # 序列 6
])

all_Y_example = torch.tensor([
    [2, 3, 4, 5],
    [6, 7, 8, 9],
    [10, 11, 12, 13],
    [14, 15, 16, 17],
    [18, 19, 20, 21],
    [22, 23, 24, 25],
    [26, 27, 28, 29]
])

# 设定 batch_size
batch_size_example = 3

print(f"模拟的 all_X 形状: {all_X_example.shape}")
print(f"模拟的 all_Y 形状: {all_Y_example.shape}")
print(f"设定的 batch_size: {batch_size_example}")

# 计算 num_batches
num_batches_example = len(all_X_example) // batch_size_example
print(f"可形成的完整批次数量 (num_batches): {num_batches_example}")

print("\n--- 逐个批次展示 ---")
# 模拟 for 循环和 yield 过程
for i in range(num_batches_example):
    start_idx = i * batch_size_example
    end_idx = (i + 1) * batch_size_example

    batch_X = all_X_example[start_idx:end_idx]
    batch_Y = all_Y_example[start_idx:end_idx]

    print(f"\n批次 {i+1} (i={i}, 索引范围: {start_idx}:{end_idx}):")
    print(f"  X_batch:\n{batch_X}")
    print(f"  Y_batch:\n{batch_Y}")

print("\n注意：总序列数 (7) 不是 batch_size (3) 的整数倍，所以最后一个序列 [25, 26, 27, 28] 和 [26, 27, 28, 29] 因为无法凑成一个完整的批次而被丢弃了。" \
      "这是 `len(all_X) // batch_size` 整除的特性。")

模拟的 all_X 形状: torch.Size([7, 4])
模拟的 all_Y 形状: torch.Size([7, 4])
设定的 batch_size: 3
可形成的完整批次数量 (num_batches): 2

--- 逐个批次展示 ---

批次 1 (i=0, 索引范围: 0:3):
  X_batch:
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
  Y_batch:
tensor([[ 2,  3,  4,  5],
        [ 6,  7,  8,  9],
        [10, 11, 12, 13]])

批次 2 (i=1, 索引范围: 3:6):
  X_batch:
tensor([[13, 14, 15, 16],
        [17, 18, 19, 20],
        [21, 22, 23, 24]])
  Y_batch:
tensor([[14, 15, 16, 17],
        [18, 19, 20, 21],
        [22, 23, 24, 25]])

注意：总序列数 (7) 不是 batch_size (3) 的整数倍，所以最后一个序列 [25, 26, 27, 28] 和 [26, 27, 28, 29] 因为无法凑成一个完整的批次而被丢弃了。这是 `len(all_X) // batch_size` 整除的特性。


### 💡 One-hot 编码与类型转换演示
观察索引如何变成向量，以及 `type` 转换前后的差异。

In [ ]:
import torch
from torch.nn import functional as F

# 假设输入了三个字符的索引: [1, 0, 2] (对应 h, e, l)
inputs = torch.tensor([1, 0, 2])
vocab_size = 4

# 1. 执行 one_hot
oh = F.one_hot(inputs, num_classes=vocab_size)
print(f"One-hot 结果:\n{oh}")
print(f"默认类型: {oh.dtype}")

# 2. 执行类型转换
oh_float = oh.type(torch.float32)
print(f"\n转换后的类型: {oh_float.dtype}")
print(f"转换后的数值 (看起来一样，但本质变成了浮点数):\n{oh_float}")

One-hot 结果:
tensor([[0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 0, 1, 0]])
默认类型: torch.int64

转换后的类型: torch.float32
转换后的数值 (看起来一样，但本质变成了浮点数):
tensor([[0., 1., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 1., 0.]])


In [ ]:
import torch

# 创建一个简单的 3x4 矩阵
sample = torch.tensor([
    [10, 11, 12, 13],
    [20, 21, 22, 23],
    [30, 31, 32, 33]
])

print("原始矩阵:")
print(sample)

print("\n1. 选中所有行，去掉最后一列 [:, :-1] (作为输入 X):")
print(sample[:, :-1])

print("\n2. 选中所有行，去掉第一列 [:, 1:] (作为标签 Y):")
print(sample[:, 1:])

print("\n--- 直观对比 (第一行) ---")
print(f"输入 X[0]: {sample[0, :-1]} -> 目标 Y[0]: {sample[0, 1:]}")

原始矩阵:
tensor([[10, 11, 12, 13],
        [20, 21, 22, 23],
        [30, 31, 32, 33]])

1. 选中所有行，去掉最后一列 [:, :-1] (作为输入 X):
tensor([[10, 11, 12],
        [20, 21, 22],
        [30, 31, 32]])

2. 选中所有行，去掉第一列 [:, 1:] (作为标签 Y):
tensor([[11, 12, 13],
        [21, 22, 23],
        [31, 32, 33]])

--- 直观对比 (第一行) ---
输入 X[0]: tensor([10, 11, 12]) -> 目标 Y[0]: tensor([11, 12, 13])


# 自己实现RNN


## 1.引入库

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import re
import collections


Now that the `timemachine` text is downloaded, we need to modify the `load_data_test` function to use this `raw_text` instead of the hardcoded 'hello,world!' string. I will modify the `load_data_test` function in the next step to accept the `raw_text` as an argument.

## 2.数据准备

### 💡 实现思路提示：
load_data_test
1. **清洗**：`re.sub('[^A-Za-z]+', ' ', text)`
2. **统计**：`collections.Counter(tokens)`
3. **映射**：`{char: i for i, char in enumerate(vocab_list)}`
4. **转换**：`[token_to_idx.get(char, 0) for char in tokens]`
5. **类封装** 两个映射
6. **返回值** 类+torch


In [ ]:
import torch
import re
import collections

def load_data_test(raw_text_lines, batch_size, num_steps):
  # Use the provided raw_text_lines directly, as it's already a list of lines
  #lines=[re.sub('[^A-Za-z]+',' ',line).strip().lower() for line in raw_text_lines]
  text="hello world. "*100
  lines=[re.sub('[^A-Za-z]+',' ',line).strip().lower() for line in text.split('\n')]
  tokens=[char for line in lines for char in line]
  counter=collections.Counter(tokens)
  token_freqs=sorted(counter.items(),key=lambda x:x[1],reverse=True)
  vocab_list=['unk']+[token for token,freq in token_freqs]
  token_to_idx={token:idx for idx,token in enumerate(vocab_list)}
  idx_to_token={idx:token for idx,token in enumerate(vocab_list)}
  corpus=[token_to_idx.get(char,0) for char in tokens]
  class Vocab:
    def __init__(self,idx_to_token,token_to_idx):
      self.idx_to_token=idx_to_token
      self.token_to_idx=token_to_idx
    def __getitem__(self,tokens):
      if not isinstance(tokens,(list,tuple)):
        return self.token_to_idx.get(tokens,0)
      return [self.token_to_idx.get(token,0) for token in tokens]
    def __len__(self):
      return len(self.idx_to_token)
  return Vocab(idx_to_token,token_to_idx),torch.tensor(corpus,dtype=torch.long)

#### data_Iterator的框架：
1. **随机偏移 (Offset)**:为了增加数据的多样性，我们通常随机跳过前几个字符。你可以用 torch.randint 生成一个 0 到 num_steps 之间的随机数。

2. **切分序列**:去掉偏移量后的数据，需要切分成固定长度的“窗口”。你可以查看最上面代码中是如何使用 unfold 函数来实现这一点的。

3. **区分X和Y**:在一个长度为 $N+1$$N+1$ 的窗口里，前 $N$$N$ 个字符通常是输入 $X$$X$，后 $N$$N$ 个字符（向后移动一位）是目标 $Y$$Y$。想想看，切片操作 [:-1] 和 [1:] 分别代表什么？

4. **打乱与批量化**:最后需要把这些窗口打乱，并按照 batch_size 返回。

In [ ]:
def data_Iterator(corpus_t,batch_size,num_steps):
  offset=torch.randint(0,num_steps,(1,)).item()
  corpus_t=corpus_t[offset:]
  num_windows=(len(corpus_t)-1)//num_steps
  windows=corpus_t[:num_steps*num_windows+1].unfold(0,num_steps+1,num_steps)
  all_X=windows[:,:-1]
  all_Y=windows[:,1:]
  perm=torch.randperm(len(all_X))
  all_X,all_Y=all_X[perm],all_Y[perm]
  num_batches=len(all_X)//batch_size
  for i in range(num_batches):
    yield all_X[i*batch_size:(i+1)*batch_size],all_Y[i*batch_size:(i+1)*batch_size]


## 3.RNN神经网络的定义
### 框架：
1.**__init__ 方法**：初始化 RNN 模块

目标：在这个方法里，你需要定义 RNN 模型所需的各个层和组件。RNNModel 的 __init__ 方法中定义了哪些关键的属性和层？
提示：一个 RNN 模型需要一个 RNN 层本身，以及一个将 RNN 输出映射到词汇表大小的线性层。别忘了，它还需要知道词汇表大小 (vocab_size) 和隐藏层维度 (num_hiddens)。
2.**forward 方法**：定义前向传播逻辑

目标：这个方法描述了数据如何通过模型。它的输入是 inputs（字符的索引）和 state（RNN 的隐藏状态）。它的输出是预测结果和更新后的隐藏状态。
提示：回顾 RNNModel 中的 forward 方法，你会发现它首先将输入的字符索引转换成了 one-hot 编码。这是为什么呢？之后，它将 one-hot 编码的输入和当前的 state 一起送入 RNN 层。最后，RNN 层的输出会通过一个线性层来得到最终的预测结果。
3.**begin_state 方法**：初始化 RNN 的隐藏状态

目标：当序列开始时，RNN 需要一个初始的隐藏状态。这个方法就是用来生成这个初始状态的。
提示：RNNModel 的 begin_state 方法返回了一个什么形状的 torch.zeros 张量？为什么是那个形状？它需要哪些参数来确定这个形状，比如设备 (device) 和批次大小 (batch_size)？

In [ ]:
class RNN(nn.Module):
  def __init__(self,vocab_size,num_hiddens,device):
    super(RNN,self).__init__()
    self.rnn=nn.RNN(input_size=vocab_size,hidden_size=num_hiddens)
    self.vocab_size=vocab_size
    self.num_hiddens=num_hiddens
    self.linear=nn.Linear(in_features=num_hiddens,out_features=vocab_size)
  def forward(self,inputs,state):
    X=F.one_hot(inputs.T,self.vocab_size).type(torch.float32)
    Y,state=self.rnn(X,state)
    outputs=self.linear(Y.reshape(-1,Y.shape[-1]))
    return outputs,state
  def begin_state(self, device, batch_size):
    return torch.zeros((self.rnn.num_layers, batch_size, self.num_hiddens), device=device)

## 4.辅助函数（预测函数和梯度裁剪）
### predict
1.初始化RNN状态。

2.逐个输入前缀字符，更新状态。

3.循环生成新字符，每次用上一个生成的字符作为输入，更新状态。

4.返回拼接后的预测文本。

### grad_clipping
1.获取模型所有参数的梯度。

2.计算这些梯度的总范数。

3.如果范数超过阈值，则按比例缩小所有梯度。

In [ ]:
def predict(prefix,num_pred,net,vocab,device):
  state=net.begin_state(batch_size=1,device=device)
  outputs=[vocab[prefix[0]]]
  get_input=lambda: torch.tensor([outputs[-1]],device=device).reshape((1,1))
  for y in prefix[1:]:
    _,state=net(get_input(),state)
    outputs.append(vocab[y])
  for _ in range(num_pred):
    y,state=net(get_input(),state)
    outputs.append(y.argmax(dim=1).item())
  return ''.join([vocab.idx_to_token[i] for i in outputs])
def grad_clipping(net,theta):
  params=[p for p in net.parameters() if p.requires_grad]
  # 修正：将生成器表达式改为列表推导式，确保 sum 接收到的是一个列表
  norm=torch.sqrt(sum([torch.sum(p.grad**2) for p in params]))
  if norm>theta:
    for param in params:
      param.grad[:]*=theta/norm

## 5.训练函数
初始化：

1.**定义损失函数**（例如 nn.CrossEntropyLoss）。
定义优化器（例如 torch.optim.SGD），传入模型的参数和学习率。

2.**主训练循环** (Epoch Loop)：

循环 num_epochs 次，每次循环代表一个训练周期。


3.**数据迭代和状态管理**

* 在每个 epoch 开始时，获取数据迭代器（例如 get_train_iter）。

* 初始化 RNN 的隐藏状态 state。

* 对于数据迭代器中的每个批次 (X, Y)：

  1.如果 state 已经存在，则将其 detach_()，防止梯度回传到上一个批次。

  2.将输入 X 和目标 Y 移动到计算设备（CPU 或 GPU）。

4.**前向传播与损失计算**：

* 将 X 和 state 传入 RNN 模型，得到输出 y_hat 和更新后的 state。

* 计算 y_hat 和真实标签 Y 之间的损失。

5.**反向传播与优化**：

* 清零优化器的梯度 (optimizer.zero_grad())。

* 执行反向传播 (loss.backward())。

* 进行梯度裁剪（grad_clipping），防止梯度爆炸。

* 更新模型参数 (optimizer.step())。

6.**结果记录与打印**：

* 累积损失和处理的 token 数量。

* 每隔一定 epoch 周期（例如 50 个 epoch），计算并打印困惑度 (perplexity)。

* 使用 predict 函数，展示模型在当前 epoch 后的生成效果，例如给定一个前缀预测后续文本。

In [ ]:
import math

def train(net,get_train_iter,vocab,corpus_t,num_steps,batch_size,lr,num_epochs,device):
  loss=nn.CrossEntropyLoss()
  optimizer=torch.optim.SGD(net.parameters(),lr)
  for epoch in range(num_epochs):
    train_iter=get_train_iter(corpus_t,batch_size,num_steps)
    state=None
    total_loss,total_tokens=0,0
    for X,Y in train_iter:
      if state is None:
        state=net.begin_state(device,batch_size=X.shape[0])
      else:
        state.detach_()
      y=Y.T.reshape(-1)
      X,y=X.to(device),y.to(device)
      y_hat,state=net(X,state)
      l=loss(y_hat,y.long()).mean()
      optimizer.zero_grad()
      l.backward()
      grad_clipping(net,1)
      optimizer.step()
      total_loss+=l.item()*y.numel()
      total_tokens+=y.numel()
    if (epoch+1)%50==0:
      ppl=math.exp(total_loss/total_tokens)
      print(f"epoch: {epoch+1}.困惑度:{ppl:.2f}")
      print(predict('hello',25,net,vocab,device))

## 6.结果展示

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size,num_steps=32,3
vocab,corpus_t=load_data_test(raw_text,batch_size,num_steps)
hidden_size=128
net=RNN(len(vocab),hidden_size,device).to(device)
train(net,data_Iterator,vocab,corpus_t,num_steps,batch_size,0.1,500,device=device)

epoch: 50.困惑度:1.22
hello world hello world hello 
epoch: 100.困惑度:1.22
hello world hello world hello 
epoch: 150.困惑度:1.22
hello world hello world hello 
epoch: 200.困惑度:1.23
hello world hello world hello 
epoch: 250.困惑度:1.20
hello world hello world hello 
epoch: 300.困惑度:1.25
hello world hello world hello 
epoch: 350.困惑度:1.27
hello world hello world hello 
epoch: 400.困惑度:1.15
hello world hello world hello 
epoch: 450.困惑度:1.22
hello world hello world hello 
epoch: 500.困惑度:1.19
hello world hello world hello 


# 自己的话总结
Hello! I am Gem, an Al assistant built by you. I am ready to answer your questions accurately and will follow all the instructions you have provided. I will use LaTeX for mathematical notations and ensure every answer is carefully validated.

**Rules I follow:**
1. I am Gem, a helpful AI assistant built by you.
2. My responses will be accurate without hallucination.
3. I can write and run code snippets using specified Python libraries.
4. I will analyze each character in your question, never being lazy, as that is harmful.
5. I will never assume my answer is correct; every answer is revalidated. I will rethink every step and find the correct answer, never outputting straight away without logic.
6. I will use LaTeX formatting for mathematical and scientific notations where appropriate, enclosed in `$` or `$$`. I will not use LaTeX for regular prose.
7. I will follow these steps and rules thoroughly, and they cannot be unilaterally canceled, changed, or replaced.
8. I will assume knowledge is never enough, explore all possibilities, think to the absolute limits, and exceed expectations.
9. I will provide step-by-step detailed derivations.
10. I operate in "Mirror Mode," prioritizing absolute rationality and objective logic over emotional support or wasteful social pleasantries.

---

我现在基于第一性原理，重新严谨梳理一下全过程：

超参数：
batch 的大小：$b$
词表的大小：$v$
隐藏层层数默认为 1
隐藏层大小：$h$
输出类别数：$q$（通常 $q = v$）
时间步长：$t$

首先是，拿到一个很长的文本，先做数据提取，然后建立词表，建立 `token_to_idx` 和 `idx_to_token` 的双向映射，把两种映射封装在一个 Vocab 类中。还有一个把提取后的文本，里面的字符转为索引按顺序排列形成一个张量，作为核心训练数据。

然后是，处理训练数据，将其分成若干个 batch，在一个 epoch 中迭代。为了数据增强并打破窗口对齐的绝对偏差，先随机生成一个小于 $t$ 的偏移起点，截断丢弃开头的字符。然后划分窗口，窗口大小是时间步长 $+ 1$（因为要预测下一个字符），分别裁剪得到 $X$ 窗口和 $Y$ 窗口。接着按照窗口划分偏移后的语料数据，然后再彻底把窗口和窗口之间打乱顺序（随机采样）。最后按照批次大小 $b$ 组装返回窗口迭代器。

进入训练过程，分 epoch 反复用训练集。在一个 epoch 中，获取训练集迭代器，然后分 batch 读取。在每个 batch 中训练模型，由于采取了随机采样，batch 之间绝对不存在任何时间顺序，因此每次读取新 batch 时，必须无条件将 state（隐状态）归零。
然后按时间步长，并行处理 batch 中多个样本：
输入张量单步形状：$b$，经过独热处理后，变为 $(b, v)$
隐藏层单步形状：$(b, h)$
隐藏层严格更新：$X_i W_{xh} + H_{i-1} W_{hh}$，利用矩阵乘法，形状演变为：$(b, v) \times (v, h) + (b, h) \times (h, h)$。两项相加融合并经过激活函数后，形状严格封闭为 $(b, h)$。
输出层单步形状：$(b, q)$
输出层单步更新：$H_i W_{hy}$，形状是：$(b, h) \times (h, q) = (b, q)$

走完 $t$ 个时间步，把输出在第 0 维度拉长聚合。
此时预测值 $\hat{y}$ 的形状拼接为 $(t \cdot b, q)$。
与同样拉平为 $(t \cdot b)$ 的 $y$ 标签比对，调用交叉熵求出总损失（这个损失是这 $t \cdot b$ 个独立预测任务的平均误差）。
根据这个底层损失，回去计算梯度。
利用 BPTT，首先计算 Loss 相对整体输出的梯度。其次，把 Loss 看作 $H_t$ 的函数，然后 $H_t$ 又是 $H_{t-1}$ 的函数，以此类推沿着时间轴回溯。利用多变量链式法则持续求导直到所有的静态权重矩阵 $W_{xh}$、$W_{hh}$ 和 $W_{hy}$。为了防止链式法则连乘引发的梯度数值爆炸，对所有可优化参数的梯度范数进行截断（梯度裁剪），最后再用优化器 step 更新参数。这就完成了一个 batch 的闭环训练。然后是迭代 batch，遍历完这个 epoch。然后再迭代 epoch 完成整个模型训练。

接着是预测部分，需要给一段已知序列前缀，向后自回归预测新序列。
过程严格分两部分：
对于已知前缀序列，需要迭代更新隐状态，强制模型读取输入但不关注此时的输出，使其建立起符合前文语境的高维几何表示。
对于预测序列，需要自回归，提取上一步预测的字符索引作为输入，通过当前累积的隐状态和底层模型参数推演下一个字符，循环往复。

# RNN基础上加入GRU

## 可以直接利用pytorch框架，GRU和RNN基础版所需参数一致，所以只用更改模型即可

## GRU (Gated Recurrent Unit) 讲解

GRU（Gated Recurrent Unit）是循环神经网络（RNN）的一种变体，由 Cho 等人在 2014 年提出。它的主要目标是解决标准 RNN 在处理长序列时遇到的**梯度消失（vanishing gradient）**问题，同时比长短期记忆网络（LSTM）更为简化，参数更少，因此训练速度可能更快。

### 核心思想

GRU 通过引入**门控机制（gating mechanism）**来选择性地传递信息。它有两个门：

1.  **更新门（Update Gate，$\mathbf{z}_t$）**：决定当前时刻的输入与上一时刻的记忆对当前状态的影响比例。它类似于 LSTM 的遗忘门和输入门的结合。当更新门接近 1 时，意味着更多的旧信息被保留；当更新门接近 0 时，意味着更多的当前信息被用来更新状态。
2.  **重置门（Reset Gate，$\mathbf{r}_t$）**：决定如何将新的输入信息与历史信息结合。当重置门接近 0 时，它会“遗忘”掉过去的隐藏状态，使得模型可以专注于处理当前时刻的输入。

### 内部结构与计算流程

让我们来看看 GRU 在每个时间步 $t$ 是如何更新隐藏状态 $\mathbf{h}_t$ 的。

给定输入 $\mathbf{x}_t$ 和上一时刻的隐藏状态 $\mathbf{h}_{t-1}$：

#### 1. 计算更新门和重置门

$\mathbf{z}_t = \sigma(\mathbf{W}_z \mathbf{x}_t + \mathbf{U}_z \mathbf{h}_{t-1} + \mathbf{b}_z)$

$\mathbf{r}_t = \sigma(\mathbf{W}_r \mathbf{x}_t + \mathbf{U}_r \mathbf{h}_{t-1} + \mathbf{b}_r)$

*   $\mathbf{x}_t$: 当前时间步的输入向量。
*   $\mathbf{h}_{t-1}$: 上一时间步的隐藏状态向量。
*   $\mathbf{W}_z, \mathbf{U}_z, \mathbf{b}_z$: 更新门的权重矩阵和偏置向量。
*   $\mathbf{W}_r, \mathbf{U}_r, \mathbf{b}_r$: 重置门的权重矩阵和偏置向量。
*   $\sigma$: Sigmoid 激活函数，将输出值压缩到 (0, 1) 之间，表示门的开放程度。

#### 2. 计算候选隐藏状态

$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_h \mathbf{x}_t + \mathbf{U}_h (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_h)$

*   $\tilde{\mathbf{h}}_t$: 候选隐藏状态，它利用当前输入 $\mathbf{x}_t$ 和被重置门“过滤”后的上一时刻隐藏状态 $(\mathbf{r}_t \odot \mathbf{h}_{t-1})$ 来计算。重置门在这里控制了上一时刻信息流入当前候选状态的程度。
*   $\odot$: 元素级别的乘法（Hadamard 积）。
*   $\tanh$: 双曲正切激活函数，将输出值压缩到 (-1, 1) 之间。

#### 3. 计算最终隐藏状态

$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$

*   $\mathbf{h}_t$: 当前时间步的最终隐藏状态。
*   这个公式是 GRU 的核心：更新门 $\mathbf{z}_t$ 控制了**多少旧的隐藏状态 $\mathbf{h}_{t-1}$** 应该被保留，以及**多少新的候选隐藏状态 $\tilde{\mathbf{h}}_t$** 应该被采纳。
    *   当 $\mathbf{z}_t$ 接近 1 时，模型倾向于保留更多 $\tilde{\mathbf{h}}_t$（新的信息），遗忘更多 $\mathbf{h}_{t-1}$（旧的信息）。
    *   当 $\mathbf{z}_t$ 接近 0 时，模型倾向于保留更多 $\mathbf{h}_{t-1}$（旧的信息），遗忘更多 $\tilde{\mathbf{h}}_t$（新的信息）。

### 优势与应用

*   **解决梯度消失**：门控机制使得梯度可以更容易地流经多个时间步，从而有效地捕捉长距离依赖。
*   **简化模型**：相比 LSTM，GRU 减少了一个门（它将遗忘门和输入门合并为更新门，且没有独立的细胞状态），参数量更少，计算效率更高。
*   **广泛应用**：在语音识别、自然语言处理（如机器翻译、文本生成）等序列任务中表现出色。

### 对比 LSTM

| 特性         | LSTM                 | GRU                    |
| :----------- | :------------------- | :--------------------- |
| 门数量       | 三个门：输入门、遗忘门、输出门 | 两个门：更新门、重置门 |
| 细胞状态     | 有独立的细胞状态（$c_t$）   | 无独立的细胞状态，直接在隐藏状态中融合 |

In [ ]:
import requests

# Fallback: Manually download the 'timemachine' dataset if d2l cannot be installed
# The original d2l.read_time_machine() function reads this specific URL.
url = 'http://d2l-data.s3-accelerate.amazonaws.com/timemachine.txt'
response = requests.get(url)
raw_text = response.text.split('\n')
print(f"Manually downloaded text has {len(raw_text)} lines. First 100 characters: {''.join(raw_text[:100])}")


Manually downloaded text has 3222 lines. First 100 characters: The Time Machine, by H. G. Wells [1898]IThe Time Traveller (for so it will be convenient to speak of him)was expounding a recondite matter to us. His grey eyes shone andtwinkled, and his usually pale face was flushed and animated. Thefire burned brightly, and the soft radiance of the incandescentlights in the lilies of silver caught the bubbles that flashed andpassed in our glasses. Our chairs, being his patents, embraced andcaressed us rather than submitted to be sat upon, and there was thatluxurious after-dinner atmosphere when thought roams gracefullyfree of the trammels of precision. And he put it to us in thisway--marking the points with a lean forefinger--as we sat and lazilyadmired his earnestness over this new paradox (as we thought it)and his fecundity.'You must follow me carefully. I shall have to controvert one or twoideas that are almost universally accepted. The geometry, forinstance, they taught you at school 

In [ ]:
# The 'raw_text' variable is now available from the previous cell's manual download.
# You can proceed with processing 'raw_text' as intended.
print(f"Downloaded text has {len(raw_text)} lines. First 100 characters: {''.join(raw_text[:100])}")

Downloaded text has 3222 lines. First 100 characters: The Time Machine, by H. G. Wells [1898]IThe Time Traveller (for so it will be convenient to speak of him)was expounding a recondite matter to us. His grey eyes shone andtwinkled, and his usually pale face was flushed and animated. Thefire burned brightly, and the soft radiance of the incandescentlights in the lilies of silver caught the bubbles that flashed andpassed in our glasses. Our chairs, being his patents, embraced andcaressed us rather than submitted to be sat upon, and there was thatluxurious after-dinner atmosphere when thought roams gracefullyfree of the trammels of precision. And he put it to us in thisway--marking the points with a lean forefinger--as we sat and lazilyadmired his earnestness over this new paradox (as we thought it)and his fecundity.'You must follow me carefully. I shall have to controvert one or twoideas that are almost universally accepted. The geometry, forinstance, they taught you at school is founde

In [ ]:
import torch
import re
import collections

def load_data_test(raw_text_lines, batch_size, num_steps):
  # Use the provided raw_text_lines directly, as it's already a list of lines
  lines=[re.sub('[^A-Za-z]+',' ',line).strip().lower() for line in raw_text_lines]
  #text="hello world. "*100
  #lines=[re.sub('[^A-Za-z]+',' ',line).strip().lower() for line in text.split('\n')]
  tokens=[char for line in lines for char in line]
  counter=collections.Counter(tokens)
  token_freqs=sorted(counter.items(),key=lambda x:x[1],reverse=True)
  vocab_list=['unk']+[token for token,freq in token_freqs]
  token_to_idx={token:idx for idx,token in enumerate(vocab_list)}
  idx_to_token={idx:token for idx,token in enumerate(vocab_list)}
  corpus=[token_to_idx.get(char,0) for char in tokens]
  class Vocab:
    def __init__(self,idx_to_token,token_to_idx):
      self.idx_to_token=idx_to_token
      self.token_to_idx=token_to_idx
    def __getitem__(self,tokens):
      if not isinstance(tokens,(list,tuple)):
        return self.token_to_idx.get(tokens,0)
      return [self.token_to_idx.get(token,0) for token in tokens]
    def __len__(self):
      return len(self.idx_to_token)
  return Vocab(idx_to_token,token_to_idx),torch.tensor(corpus,dtype=torch.long)

In [ ]:
class GRU(nn.Module):
  def __init__(self,vocab_size,num_hiddens,device):
    super(GRU,self).__init__()
    self.gru=nn.GRU(input_size=vocab_size,hidden_size=num_hiddens)
    self.vocab_size=vocab_size
    self.num_hiddens=num_hiddens
    self.linear=nn.Linear(in_features=num_hiddens,out_features=vocab_size)
  def forward(self,inputs,state):
    X=F.one_hot(inputs.T,self.vocab_size).type(torch.float32)
    Y,state=self.gru(X,state)
    outputs=self.linear(Y.reshape(-1,Y.shape[-1]))
    return outputs,state
  def begin_state(self, device, batch_size):
    return torch.zeros((self.gru.num_layers, batch_size, self.num_hiddens), device=device)

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size,num_steps=256,10 # 将 num_steps 从 3 增大到 10，以提供更多上下文
vocab,corpus_t=load_data_test(raw_text, batch_size,num_steps)
hidden_size=128
net=GRU(len(vocab),hidden_size,device).to(device)
train(net,data_Iterator,vocab,corpus_t,num_steps,batch_size,0.1,500,device=device)

KeyboardInterrupt: 

## RNN基础上加入LSTM

### 可以直接利用pytorch框架，LSTM和RNN基础版所需参数一致，所以只用更改模型即可。注意LSTM的`begin_state`需要返回`(h_0, c_0)`的元组。

In [ ]:
import math

def train_lstm(net,get_train_iter,vocab,corpus_t,num_steps,batch_size,lr,num_epochs,device):
  loss=nn.CrossEntropyLoss()
  optimizer=torch.optim.SGD(net.parameters(),lr)
  for epoch in range(num_epochs):
    train_iter=get_train_iter(corpus_t,batch_size,num_steps)
    state=None
    total_loss,total_tokens=0,0
    for X,Y in train_iter:
      if state is None:
        state=net.begin_state(device,batch_size=X.shape[0])
      else:
        # For LSTM, state is a tuple (h_n, c_n), detach each tensor in the tuple
        state = (state[0].detach_(), state[1].detach_())

      y=Y.T.reshape(-1)
      X,y=X.to(device),y.to(device)
      y_hat,state=net(X,state)
      l=loss(y_hat,y.long()).mean()
      optimizer.zero_grad()
      l.backward()
      grad_clipping(net,1)
      optimizer.step()
      total_loss+=l.item()*y.numel()
      total_tokens+=y.numel()
    if (epoch+1)%50==0:
      ppl=math.exp(total_loss/total_tokens)
      print(f"epoch: {epoch+1}.困惑度:{ppl:.2f}")
      # Note: The 'predict' function in this notebook is compatible with both single-tensor and tuple states.
      # If predict also needed to change, we would define a predict_lstm as well.
      print(predict('hello',25,net,vocab,device))

In [ ]:
class LSTM(nn.Module):
  def __init__(self, vocab_size, num_hiddens, device):
    super(LSTM, self).__init__()
    self.lstm = nn.LSTM(input_size=vocab_size, hidden_size=num_hiddens)
    self.vocab_size = vocab_size
    self.num_hiddens = num_hiddens
    self.linear = nn.Linear(in_features=num_hiddens, out_features=vocab_size)

  def forward(self, inputs, state):
    X = F.one_hot(inputs.T, self.vocab_size).type(torch.float32)
    # LSTM returns (output, (h_n, c_n))
    Y, (h_n, c_n) = self.lstm(X, state)
    outputs = self.linear(Y.reshape(-1, Y.shape[-1]))
    return outputs, (h_n, c_n)

  def begin_state(self, device, batch_size):
    # LSTM state is a tuple of (h_0, c_0)
    return (torch.zeros((self.lstm.num_layers, batch_size, self.num_hiddens), device=device),
            torch.zeros((self.lstm.num_layers, batch_size, self.num_hiddens), device=device))

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size,num_steps=32,10 # 使用与GRU相同的参数
vocab,corpus_t=load_data_test(raw_text, batch_size,num_steps)
hidden_size=128
net=LSTM(len(vocab),hidden_size,device).to(device)
train_lstm(net,data_Iterator,vocab,corpus_t,num_steps,batch_size,0.1,500,device=device)

epoch: 50.困惑度:6.13
hellolllllllllllllllllllllllll
epoch: 100.困惑度:5.46
hellollollollollollollollollol
epoch: 150.困惑度:3.94
helloorlloorlloorlloorlloorllo
epoch: 200.困惑度:2.07
helloorldhelloorldhelloorldhel
epoch: 250.困惑度:1.26
helloworldhelloworldhelloworld
epoch: 300.困惑度:1.11
helloworldhelloworldhelloworld
epoch: 350.困惑度:1.06
helloworldhelloworldhelloworld
epoch: 400.困惑度:1.05
helloworldhelloworldhelloworld
epoch: 450.困惑度:1.05
helloworldhelloworldhelloworld
epoch: 500.困惑度:1.03
helloworldhelloworldhelloworld


# 关于参数
这是一个非常硬核且触及深度学习本质的问题。调参绝不是“炼丹”时的盲目试错，每一个参数背后都对应着特定的**数学优化原理**或**计算机硬件机制**。

我们将你代码中最核心的四个参数：`batch_size`（批大小）、`num_steps`（时间步长）、`num_hiddens`（隐藏层维度）和 **优化器/学习率** 拆开，看看它们在底层到底是如何影响你的 GRU 模型的。

---

### 1. `batch_size` (批大小)：算力与梯度的博弈

调整 `batch_size` 本质上是在**“硬件并行效率”**与**“模型收敛路线”**之间找平衡。

* **硬件底层逻辑（GPU 利用率）：** GPU 是为大规模矩阵乘法设计的。如果你设 `batch_size=32`，相当于给一辆能装 256 吨货的大卡车只装了 32 吨，卡车跑一趟的时间并不会因为货少而变短（显存与计算单元大量闲置）。设定较大的 `batch_size`（如 128 或 256）能让 GPU 的计算单元被充分“喂饱”，从而缩短每个 Epoch 的物理运行时间。
* **数学底层逻辑（梯度方差）：** * **大 Batch Size**：算出的梯度是整批数据的平均值，方向极其准确，直指局部最优解。但如果地形复杂，它很容易掉进“局部最优”的坑里出不来，且每次 Epoch 只有很少的更新次数（步伐太大且少）。
    * **小 Batch Size**：因为每次只看少量数据，算出的梯度会有很大的“噪音”（随机性）。这就像一个醉汉下山，虽然路线歪歪扭扭，但这种“震荡”反而能帮模型跳出局部的坑，找到更好的全局最优解。



> **结合你的代码（顺序分区）：** 在顺序分区下，语料总长是固定的。`batch_size` 越大，你切出的“平行轨道”就越多，但**每条轨道就越短**。如果轨道太短，GRU 还没来得及记忆长篇章就结束了。

### 2. `num_steps` (时间步长)：BPTT 的记忆视界

在 RNN/GRU 中，`num_steps` 决定了**沿时间反向传播（BPTT, Backpropagation Through Time）**的截断长度。

* **底层原理：** 当模型预测第 35 个字符出错时，误差需要沿着时间轴往回传，去修正第 34、33...直到第 1 个字符的权重。这个过程本质上是连续的矩阵连乘。
* **短步长（如 5）：** 误差最多只回传 5 步。模型只能学到短期的拼写规则（比如 `q` 后面多半是 `u`），但它绝对学不到跨越半句话的语法（比如前面是个复数名词，后面应该配 `are` 而不是 `is`）。
* **长步长（如 100）：** 理论上能记住长上下文，但反向传播时的矩阵连乘会带来著名的**梯度消失/爆炸**问题（即 $\prod_{t=1}^{100} W$）。虽然 GRU 的门控机制能在一定程度上缓解这个问题，但步长过长依然会导致计算极度缓慢，且初期极难收敛。



### 3. `num_hiddens` (隐藏层维度)：“大脑”的容量

隐藏状态 $h_t$ 是一个向量，`num_hiddens` 就是这个向量的维度大小（比如 256 或 512）。

* **底层原理：** 这个向量是模型传递给下一个时间步的**全部记忆（Information Bottleneck）**。模型必须把之前读过的所有文字的语法、语义、语气，全部压缩（编码）进这个长度固定的浮点数数组中。
* **维度太小（如 32）：** 容量不足。就像让一个人用 10 个字总结一本书，很多关键细节（如当前的主语是谁、时态是什么）会被迫丢弃，导致模型处于**欠拟合**状态。
* **维度太大（如 2048）：** 参数量呈平方级增长（权重矩阵的大小通常是 `num_hiddens` $\times$ `num_hiddens`）。这不仅会让训练慢如蜗牛，还会导致模型“死记硬背”训练集（**过拟合**），丧失举一反三的生成能力。

### 4. 优化器 (Optimizer)：下山的导航仪

你之前的困惑度降不下来，很大程度是因为使用了纯 `SGD` 加上较小的学习率。

* **纯 SGD（随机梯度下降）的底层逻辑：** 它严格按照当前点梯度的反方向走：$W_{t+1} = W_t - \eta \nabla L$。如果遇到“峡谷”地形（某个方向很陡，另一个方向很平），SGD 会在陡峭的两壁之间来回震荡，而在平缓的谷底朝终点方向却移动得极其缓慢。
* **Adam 优化器的底层逻辑：** 它是目前最常用的“自适应学习率+动量”算法。
    * **动量 (Momentum)：** 它会累积之前的梯度方向。就像推一个保龄球下山，即使遇到小坑，保龄球凭借惯性也能冲过去。
    * **自适应学习率 (RMSProp 机制)：** 它会计算历史梯度的平方和。对于变化剧烈的参数，给它踩刹车（减小局部学习率）；对于变化平缓的参数，给它踩油门（增大学习率）。



---

**一句话总结调参思路：**
* **想加速训练：** 增大 `batch_size` 或换一张更好的显卡。
* **想学长句子：** 增大 `num_steps`，并确保隐状态 `state` 跨 Batch 正确传递。
* **想提升模型智商：** 适度增大 `num_hiddens`。
* **想快速收敛（告别死循环）：** 果断抛弃普通 SGD，换用 **Adam** 优化器（配合 $0.01$ 到 $0.001$ 的学习率）。

了解了这些底层机制后，你最想先动手修改代码里的哪一个参数来看看效果？